In [1]:
import Student
!pip install wilds matplotlib torch torchvision terratorch


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from wilds import get_dataset
import pandas.core.tools.datetimes as _pd_dt_module
import matplotlib.pyplot as plt
import pandas as pd


my_root_path = "/mnt/windows/Users/kevin/Downloads/"

_true_orig_to_datetime = _pd_dt_module.to_datetime

def _patched_to_datetime(*args, **kwargs):
    # Always force ISO8601 — overrides missing OR incorrect format
    kwargs['format'] = 'ISO8601'
    return _true_orig_to_datetime(*args, **kwargs)

pd.to_datetime = _patched_to_datetime

#never use dataset, but use the metadata(it contains all IMAGES)
dataset = get_dataset(dataset="fmow", root_dir=my_root_path, download=False)

# Restore
pd.to_datetime = _true_orig_to_datetime

# Grab the pre-built, rich pandas DataFrame directly from the dataset
df = dataset.metadata.copy()

# Extract the timestamp safely by explicitly defining the ISO8601 format
df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601')
df['year_extracted'] = df['timestamp'].dt.year
#Find the exact column index for 'region' in the metadata_array
region_idx = dataset.metadata_fields.index('region')

# Retrieve the mapping list from the dataset (e.g., ['Asia', 'Europe', ...])
region_names_list = dataset.metadata_map['region']

df['region_names'] = df['region'].map(lambda x: region_names_list[int(x)])

In [3]:
# ---------------------------------------------------------
#  Simplified DataLoader
# ---------------------------------------------------------
import torch
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
from terratorch.models.backbones.terramind.model.terramind_register import v1_pretraining_mean, v1_pretraining_std

def get_input(idx):
        """
        Returns x for a given idx.
        """
        img = Image.open(Path(my_root_path + 'fmow_v1.1') / 'images' / f'rgb_img_{idx}.png').convert('RGB')
        return img



# 1. Fetch the 8-bit RGB specific statistics
tm_rgb_mean = v1_pretraining_mean['untok_sen2rgb@224']
tm_rgb_std = v1_pretraining_std['untok_sen2rgb@224']

# 2. Scale the stats down to 0.0-1.0 to match T.ToTensor()
SCALED_MEAN = [x / 255.0 for x in tm_rgb_mean]
SCALED_STD = [x / 255.0 for x in tm_rgb_std]

print(f"Scaled RGB Mean: {SCALED_MEAN}")
print(f"Scaled RGB Std:  {SCALED_STD}")

# 3. Create the Transform Pipeline
terramind_transform = T.Compose([
    T.ToTensor(),  # Scales 0-255 pixels down to 0.0-1.0
    T.Normalize(mean=SCALED_MEAN, std=SCALED_STD) # Perfectly standardizes the data
])

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
resnet_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD)
])




def get_tensor_images(numberOfImages):
    lista_tensori = []
    for i in range(0,numberOfImages):
        img = get_input(i)
        img_t = terramind_transform(img)
        lista_tensori.append(img_t)

    batch_immagini = torch.stack(lista_tensori,dim = 0)
    return batch_immagini


/home/kevinthomaj/PyCharmMiscProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Scaled RGB Mean: [0.3422392156862745, 0.31737647058823526, 0.26143921568627454]
Scaled RGB Std:  [0.23045882352941177, 0.18691372549019605, 0.16718039215686276]


In [4]:
import FoundationModel
# %%
# ---------------------------------------------------------
# CLUSTERING DEGLI EMBEDDINGS (Creazione dei Concepts)
# ---------------------------------------------------------
import torch
import numpy as np
from tqdm import tqdm
import gc


#Random Sampling with similar number of images per class
print("--- Inizio Estrazione Embeddings e Clustering ---")

samples_per_class = 32
# 1. Create an empty list to hold our exact row numbers
sample_indices = []

# 2. Iterate through each class group and sample safely
for category, group in df.groupby('category'):
    n_to_sample = min(len(group), samples_per_class)
    sampled_group = group.sample(n=n_to_sample, random_state=42)
    sample_indices.extend(sampled_group.index.tolist())

# 3. Create df_sample by plucking those exact indices from the original dataframe
df_sample = df.loc[sample_indices].copy()


--- Inizio Estrazione Embeddings e Clustering ---


In [5]:
import os
print("Your files will be saved here:", os.getcwd())

Your files will be saved here: /home/kevinthomaj/PyCharmMiscProject


In [6]:
import os
import torch
import numpy as np
from tqdm import tqdm
import gc

# --- File Paths for Caching ---
EMBEDDINGS_FILE = 'terramind_embeddings.npy'
INDICES_FILE = 'sampled_indices.npy'

# --- Device Setup ---
# Ensure you are using GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if os.path.exists(EMBEDDINGS_FILE) and os.path.exists(INDICES_FILE):
    print("Found saved embeddings! Loading directly from disk...")

    # 1. Load Numpy arrays
    embeddings_matrix = np.load(EMBEDDINGS_FILE)
    sample_indices = np.load(INDICES_FILE).tolist()

    # 2. Reconstruct df_sample perfectly
    df_sample = df.loc[sample_indices].copy()

    print(f"Loaded embeddings matrix shape: {embeddings_matrix.shape}")

else:
    print("No saved data found. Starting Terramind extraction loop...")

    # 1. Load the model and move to device
    fm_model = FoundationModel.FoundationModel()
    fm_model = fm_model.to(device)

    batch_size = 128
    all_embeddings = []
    full_resnet_dataset_list = []

    with torch.no_grad():
        for i in tqdm(range(0, len(sample_indices), batch_size), desc="Estrazione feature FM"):
            batch_idx = sample_indices[i:i+batch_size]

            batch_imgs = []
            batch_imgs_resnet = []
            for idx in batch_idx:
                img = get_input(idx)
                img_t = terramind_transform(img)
                batch_imgs.append(img_t)

            # Stack tensor
            input_tensor = torch.stack(batch_imgs, dim=0).to(device)

            # Forward pass nel Foundation Model
            features = fm_model(input_tensor)

            # Sposta su CPU prima di salvare in numpy/liste (per non saturare la VRAM)
            all_embeddings.append(features.cpu().numpy())

            # Cleanup memoria
            del input_tensor
            del features
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    # 2. Concateniamo tutti i batch
    embeddings_matrix = np.concatenate(all_embeddings, axis=0)

    # 3. Save to disk for the next session
    print("Salvataggio su disco in corso...")
    np.save(EMBEDDINGS_FILE, embeddings_matrix)
    np.save(INDICES_FILE, np.array(sample_indices))

    print("Estrazione e salvataggio completati!")
    print(f"Forma della matrice degli embeddings: {embeddings_matrix.shape}")

No saved data found. Starting Terramind extraction loop...


2026-04-15 17:21:37,119 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-base/resolve/main/TerraMind_v1_base.pt "HTTP/1.1 302 Found"
Estrazione feature FM:   0%|          | 1/474 [00:45<5:57:46, 45.38s/it]


KeyboardInterrupt: 

In [ ]:
# %%
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
import pandas as pd

# 1. Create the embeddings dataframe
df_emb = pd.DataFrame(embeddings_matrix)

# 2. Since your dataframe already contains the string names,
# simply assign them directly to a new column!
df_emb['category_name'] = df_sample['category'].values

# 3. Group by the string names to get the centroids
class_centroids = df_emb.groupby('category_name').mean()

# Your index now automatically contains the correct string labels
name_classes = list(class_centroids.index)
matrix_centroids = class_centroids.values

print(f"Centroids matrix shape: {matrix_centroids.shape}")

# Compute linkage
linkage_data = linkage(class_centroids, method='ward', metric='euclidean')

# ---------------------------------------------------------
# Plotting the Dendrogram
# ---------------------------------------------------------
plt.figure(figsize=(20, 10))

dendrogram(
    linkage_data,
    labels=name_classes,
    leaf_rotation=90,
    leaf_font_size=10
)

plt.title('Hierarchical Clustering Dendrogram (62 FMoW Classes)', fontsize=16)
plt.xlabel('Category Name', fontsize=14)
plt.ylabel('Distance', fontsize=14)
plt.tight_layout()

plt.show()

In [ ]:
from scipy.cluster.hierarchy import fcluster

# 1. Cut the hierarchical tree to form exactly 6 clusters
# t=6 specifies the max number of clusters, criterion='maxclust' enforces this.
cluster_labels = fcluster(linkage_data, t=6, criterion='maxclust')

# 2. Create a dictionary mapping the original 62 classes to the 6 new macro-clusters
# Example output: {'airport': 1, 'barn': 3, 'bridge': 1...}
class_to_macro_cluster = dict(zip(name_classes, cluster_labels))

# 3. Apply this mapping back to my sample dataframe
df_sample['macro_class'] = df_sample['category'].map(class_to_macro_cluster)

# Verify the distribution of the new 6 classes
print("Distribution of images across the 6 new macro-classes in the full dataset:")
print(df_sample['macro_class'].value_counts())


In [ ]:
import torch
import torch.nn as nn
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=6):
        super(SimpleMLP, self).__init__()
        self.layer_stack = nn.Sequential(
            # Input to Hidden
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # Optional: Dropout helps significantly with 64 samples/class
            #nn.Dropout(0.2),
            # Hidden to Output
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.layer_stack(x)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# --- 1. Functions for Modular Phases ---

def train_one_epoch(model, loader, optimizer, criterion, device):
    """Handles the full training pass for one epoch."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # Using argmax for cleaner syntax
        predicted = torch.argmax(outputs, dim=1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    return running_loss / len(loader), 100 * correct / total

def evaluate(model, loader, criterion, device):
    """Handles evaluation (Validation or Testing)."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            running_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    return running_loss / len(loader), 100 * correct / total

# --- 2. Setup & Data Loading ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

le = LabelEncoder()
y = le.fit_transform(df_sample['category'])
# Split Data
X_train, X_test, y_train, y_test = train_test_split(
    embeddings_matrix, y, test_size=0.2, stratify=y, random_state=42
)

# Convert to Loaders
train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                        torch.tensor(y_train, dtype=torch.long)),
                          batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                       torch.tensor(y_test, dtype=torch.long)),
                         batch_size=32, shuffle=False)

# Initialize
model = SimpleMLP(input_dim=X_train.shape[1], hidden_dim=128, num_classes=62).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- 3. The Main Execution Loop ---

epochs = 100
history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(epochs):
    # Training Phase
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validation Phase (Testing inside the loop)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    # Log History
    history['train_loss'].append(train_loss)
    history['test_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(val_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")

# --- 4. Final Test Phase ---

print("\n--- Final Evaluation ---")
final_test_loss, final_test_acc = evaluate(model, test_loader, criterion, device)
print(f"Final Test Accuracy: {final_test_acc:.2f}%")